# 00 — How LLMs Actually Work: Mental Models for AI Engineers

> **Orbit -1 (Foundations)** · **Difficulty**: Beginner · **Reading time**: ~25 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/foundations/00_how_llms_work.ipynb)

**What you will learn**:
- The transformer as a next-token prediction machine
- Why "predicting the next word" produces intelligent behavior
- The generation pipeline: tokenize → embed → transform → predict → sample → decode
- What temperature, top-p, and top-k actually control
- Mental models that make prompt engineering, RAG, and finetuning intuitive

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Loading our model**
- Understand **What is a language model?**
- Understand **The generation pipeline, step by step**
- Understand **Why next-token prediction produces intelligence**
- Understand **The transformer: a black box tour**


In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" "accelerate>=0.30.0" \
    "bitsandbytes>=0.43.0" "torch>=2.1.0" "numpy>=1.24.0" \
    "matplotlib>=3.7.0"

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Mount Google Drive for model caching
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
CACHE_DIR = "/content/drive/MyDrive/models"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = CACHE_DIR

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✓ GPU: {gpu_name} — {vram_gb:.1f} GB VRAM")
else:
    print("⚠ No GPU detected — notebook will run slower but still works")

print(f"✓ PyTorch {torch.__version__} on {device}")
print("✓ All packages installed — ready to go")

---

## Loading our model

We'll use **Qwen3-0.6B** throughout this notebook — a 0.6-billion-parameter
language model that fits comfortably on a T4 GPU (~1.2 GB VRAM in float16).
It's small enough to inspect internals yet capable enough to show real
language understanding.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    cache_dir=CACHE_DIR,
)
model.eval()

# Report VRAM usage
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"✓ Model loaded — {allocated:.2f} GB VRAM used")
else:
    print("✓ Model loaded on CPU")

print(f"Vocabulary size : {model.config.vocab_size:,}")
print(f"Hidden size     : {model.config.hidden_size}")
print(f"Layers          : {model.config.num_hidden_layers}")
print(f"Attention heads : {model.config.num_attention_heads}")

---

## 1 — What is a language model?

Strip away the hype and a language model is a **function**:

```
f(sequence of tokens) → probability distribution over next token
```

That's it. The model takes every token it has seen so far and outputs a
score (called a **logit**) for *every* token in its vocabulary. A vocabulary
of 151,936 tokens means the model outputs 151,936 numbers — one per
possible next token.

We convert those raw logits into probabilities with the **softmax** function:

$$P(\text{token}_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

where $z_i$ is the logit for token $i$. Higher logit → higher probability.

Let's make this concrete.

In [ ]:
# Feed a prompt and inspect raw logits
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

# Logits for the LAST position — the model's prediction for the next token
logits = outputs.logits[0, -1, :]  # shape: (vocab_size,)
print(f"Logit tensor shape: {logits.shape}")
print(f"Min logit: {logits.min().item():.2f}  |  Max logit: {logits.max().item():.2f}")
print(f"These are raw, unnormalized scores — not probabilities yet.")

### The probability distribution over vocabulary

We apply softmax to convert logits into probabilities. Then we can see which
tokens the model thinks are most likely to come next. For *"The capital of
France is"*, the model should overwhelmingly predict *" Paris"* — because it
has compressed world knowledge into its weights during training.

In [ ]:
# Convert logits to probabilities and show the top-20 predicted tokens
probs = F.softmax(logits.float(), dim=-1)

top_k_values, top_k_indices = torch.topk(probs, k=20)

tokens = [tokenizer.decode(idx.item()) for idx in top_k_indices]
probabilities = top_k_values.cpu().numpy()

# Print as a table
print(f"Top-20 predictions for: \"{prompt} ___\"\n")
print(f"{"Rank":<6}{"Token":<20}{"Probability":>12}")
print("─" * 38)
for i, (tok, p) in enumerate(zip(tokens, probabilities), 1):
    print(f"{i:<6}{repr(tok):<20}{p:>11.4%}")

In [ ]:
# Visualize as a bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(tokens)-1, -1, -1), probabilities[::-1], color="#4C72B0")
ax.set_yticks(range(len(tokens)-1, -1, -1))
ax.set_yticklabels([repr(t) for t in tokens], fontsize=10)
ax.set_xlabel("Probability")
ax.set_title(f'Next-token probabilities for: "{prompt} ___"')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nThe model assigns {probabilities[0]:.1%} probability to the top token.")
print(f"The remaining {1 - probabilities[0]:.1%} is spread across {model.config.vocab_size:,} other tokens.")

---

## 2 — The generation pipeline, step by step

When you call `model.generate()` (or hit "Send" in ChatGPT), six steps
happen in sequence. Understanding each step is the single most useful
mental model for AI engineering.

```
text → [Tokenize] → token IDs → [Embed] → vectors
  → [Transform] → contextualized vectors → [Predict] → logits
  → [Sample] → next token ID → [Decode] → text
```

Let's walk through each step with real values.

### Step 1: Tokenization — text → token IDs

The tokenizer splits raw text into **sub-word tokens** and maps each to an
integer ID. This is the vocabulary the model actually works with — not
characters, not whole words, but something in between.

We'll cover tokenization in depth in
[01_tokenization_deep_dive.ipynb](01_tokenization_deep_dive.ipynb). For now,
just see the mapping:

In [ ]:
# Step 1: Tokenization
prompt = "Large language models predict the next token"

token_ids = tokenizer.encode(prompt)
tokens_list = tokenizer.convert_ids_to_tokens(token_ids)

print(f"Text:       {repr(prompt)}")
print(f"Token IDs:  {token_ids}")
print(f"Tokens:     {tokens_list}")
print(f"Num tokens: {len(token_ids)}")

# Show the mapping token-by-token
print(f"\n{"Token":<20}{"ID":>8}")
print("─" * 28)
for tok, tid in zip(tokens_list, token_ids):
    print(f"{repr(tok):<20}{tid:>8}")

### Step 2: Embedding — token IDs → vectors

Each token ID is an index into an **embedding table** — a giant matrix of
shape `(vocab_size, hidden_size)`. Looking up a token ID retrieves a dense
vector that represents that token's "meaning" in a high-dimensional space.

At this stage, each vector knows only about its own token — no context yet.
That's what the transformer layers are for.

In [ ]:
# Step 2: Embedding lookup
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

# Access the embedding layer directly
embedding_layer = model.get_input_embeddings()
embeddings = embedding_layer(input_ids)

print(f"Input IDs shape : {input_ids.shape}        # (batch, seq_len)")
print(f"Embeddings shape: {embeddings.shape}  # (batch, seq_len, hidden_size)")
print(f"\nEach of the {input_ids.shape[1]} tokens is now a {embeddings.shape[2]}-dimensional vector.")

# Peek at the first token's embedding (first 10 dimensions)
first_vec = embeddings[0, 0, :10].detach().cpu().float().numpy()
print(f"\nFirst 10 dims of token \"{tokens_list[0]}\":  {np.round(first_vec, 4)}")

### Step 3: Transformer layers — the black box that contextualizes

The embeddings pass through the model's transformer layers (Qwen3-0.6B has
28 layers). Each layer applies **self-attention** and a **feed-forward
network**, progressively enriching each vector with context from every
other token in the sequence.

Before the transformer: the vector for *"bank"* is the same whether the
sentence is about rivers or money.

After the transformer: the vector for *"bank"* carries the context of the
entire sentence. This is the magic step — we'll open the black box in
[05_attention_and_context.ipynb](05_attention_and_context.ipynb).

For now, let's run the full forward pass and see that the output changes:

In [ ]:
# Step 3: Run through all transformer layers
with torch.no_grad():
    outputs = model(input_ids, output_hidden_states=True)

# Compare embedding (layer 0) vs final hidden state (last layer)
initial = outputs.hidden_states[0][0, 0, :10].cpu().float().numpy()
final = outputs.hidden_states[-1][0, 0, :10].cpu().float().numpy()

print(f"Layers in model: {len(outputs.hidden_states) - 1}")  # -1 for embedding layer
print(f"\nFirst token \"{tokens_list[0]}\":")
print(f"  Before transformer (first 10 dims): {np.round(initial, 4)}")
print(f"  After transformer  (first 10 dims): {np.round(final, 4)}")

# How much did the representation change?
cos_sim = F.cosine_similarity(
    outputs.hidden_states[0][0, 0].unsqueeze(0).float(),
    outputs.hidden_states[-1][0, 0].unsqueeze(0).float()
).item()
print(f"\n  Cosine similarity (before vs after): {cos_sim:.4f}")
print(f"  The vector was substantially transformed by the 28 layers.")

### Step 4: Prediction — final hidden state → logits → probabilities

After the transformer layers, a **language model head** (a single linear
layer) projects each hidden state back to vocabulary size, producing one
logit per token in the vocabulary. We care about the logits at the *last*
position — those predict what comes next.

In [ ]:
# Step 4: Logits → probabilities
logits = outputs.logits[0, -1, :]  # (vocab_size,)
print(f"Logits shape: {logits.shape}  — one score per vocabulary token")

probs = F.softmax(logits.float(), dim=-1)
print(f"Probabilities sum: {probs.sum().item():.6f}  — they form a valid distribution")

# Top 5 next-token predictions
top5_probs, top5_ids = torch.topk(probs, 5)
print(f"\nTop-5 next tokens for: \"{prompt} ___\"")
for p, idx in zip(top5_probs, top5_ids):
    print(f"  {repr(tokenizer.decode(idx.item())):<15} {p.item():.4%}")

### Step 5: Sampling — pick the next token

Now we have a probability distribution. How do we pick a token?

- **Greedy decoding**: always pick the highest-probability token. Deterministic
  but repetitive.
- **Random sampling**: draw from the distribution. Creative but sometimes
  incoherent.

In practice, we use **temperature**, **top-k**, and **top-p** to control the
tradeoff. We'll explore these in Section 5.

In [ ]:
# Step 5: Greedy vs random sampling
greedy_token_id = torch.argmax(probs).item()
print(f"Greedy pick:  {repr(tokenizer.decode(greedy_token_id))}  (highest prob)")

# Random sampling — draw 5 times
sampled_ids = torch.multinomial(probs, num_samples=5, replacement=True)
sampled_tokens = [tokenizer.decode(sid.item()) for sid in sampled_ids]
print(f"Random draws: {[repr(t) for t in sampled_tokens]}")

### Step 6: Decode — token ID → text

Finally, we convert the chosen token ID back to text. Then we append it to
the input and repeat the entire pipeline to generate the *next* token. This
is called **autoregressive generation** — the model's output becomes its
own input, one token at a time.

In [ ]:
# Step 6: Decode and show the full autoregressive loop
print("=== Full generation loop (10 tokens, greedy) ===\n")
generated_ids = input_ids.clone()

for step in range(10):
    with torch.no_grad():
        out = model(generated_ids)
    next_logits = out.logits[0, -1, :]
    next_id = torch.argmax(next_logits).unsqueeze(0).unsqueeze(0)
    generated_ids = torch.cat([generated_ids, next_id], dim=-1)

    decoded = tokenizer.decode(next_id[0], skip_special_tokens=True)
    print(f"  Step {step+1:>2}: picked {repr(decoded):<12} → "
          f"\"{tokenizer.decode(generated_ids[0], skip_special_tokens=True)}\"")

print(f"\nFinal output: \"{tokenizer.decode(generated_ids[0], skip_special_tokens=True)}\"")

---

## 3 — Why next-token prediction produces intelligence

It seems absurd: a model trained *only* to predict the next word becomes
capable of reasoning, translation, coding, and creative writing. How?

### The compression hypothesis

To predict well, the model must **compress** the training data into its
weights. That compression forces it to learn the *rules* behind the data
rather than memorize the data itself:

- To predict the next word in a news article about France, it must encode
  the fact that Paris is the capital.
- To predict the next token in Python code, it must learn syntax rules,
  variable scoping, and common design patterns.
- To predict the next word in a dialogue, it must model social norms,
  speaker intent, and conversational structure.

**Next-token prediction is a proxy objective.** The model doesn't "try to
learn world knowledge" — it tries to minimize prediction error, and world
knowledge is what it needs to do that.

In [ ]:
# The model encodes world knowledge as a side effect of prediction
knowledge_probes = [
    "The chemical formula for water is",
    "def fibonacci(n):\n    if n <=",
    "The theory of relativity was proposed by Albert",
    "To convert Celsius to Fahrenheit, multiply by 9/5 and add",
    "In chess, the opening move e4 is called the",
]

print("What the model 'knows' from predicting the next token:\n")
for probe in knowledge_probes:
    ids = tokenizer(probe, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model(ids)
    top_id = torch.argmax(out.logits[0, -1, :]).item()
    predicted = tokenizer.decode(top_id)
    print(f'  "{probe}" → {repr(predicted)}')

### The scaling insight

Small models learn surface patterns — common phrases, simple grammar. As
models scale (more parameters, more data, more compute), they learn
progressively deeper structure:

| Scale | What the model learns |
|-------|----------------------|
| ~100M params | Common phrases, basic grammar |
| ~1B params | World facts, simple reasoning |
| ~10B params | Multi-step reasoning, code generation |
| ~100B+ params | Instruction following, nuanced analysis |

Our 0.6B model sits in the 1B-ish tier — good enough to demonstrate facts
and basic patterns, but you'll see it stumble on harder reasoning tasks.
That's expected and instructive.

In [ ]:
# Show limits: the small model struggles with harder reasoning
hard_probes = [
    "If I have 3 apples and give away 1, I have",
    "The word 'happy' is the opposite of",
]

print("Easy vs harder predictions with a 0.6B model:\n")
for probe in hard_probes:
    ids = tokenizer(probe, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model(ids)
    # Show top-3 predictions
    top3_probs, top3_ids = torch.topk(
        F.softmax(out.logits[0, -1, :].float(), dim=-1), 3
    )
    preds = [(tokenizer.decode(i.item()), p.item()) for i, p in zip(top3_ids, top3_probs)]
    print(f'  "{probe}___"')
    for tok, p in preds:
        print(f"    {repr(tok):<12} ({p:.2%})")
    print()

---

## 4 — The transformer: a black box tour

We won't build a transformer from scratch here — that's
[05_attention_and_context.ipynb](05_attention_and_context.ipynb). Instead,
let's build the practitioner's mental model.

### The core idea: attention

Each transformer layer has a **self-attention** mechanism that lets every
token "look at" every other token and decide how much to incorporate their
information. Think of it as:

> *"For each word I'm processing, which other words in the sentence should
> I pay attention to?"*

For the sentence *"The cat sat on the mat because it was tired"*:
- When processing *"it"*, attention learns to focus on *"cat"* (the referent)
- When processing *"tired"*, attention focuses on *"cat"* and *"it"*

### Why this matters for practitioners

Attention is why **context matters so much** in prompts. Every token you
include influences every other token's representation. This is why:
- Prompt engineering works (different prefixes → different attention patterns)
- RAG works (injected context shifts attention toward relevant facts)
- Context windows have limits (attention is O(n²) in sequence length)

In [ ]:
# Extract attention patterns from the model
sentence = "The cat sat on the mat because it was tired"
inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
tokens_for_display = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])

with torch.no_grad():
    attn_output = model(**inputs, output_attentions=True)

# Attention shape: (batch, num_heads, seq_len, seq_len)
# Pick the last layer, average across attention heads
last_layer_attn = attn_output.attentions[-1][0]  # (num_heads, seq, seq)
avg_attn = last_layer_attn.mean(dim=0).cpu().float().numpy()  # (seq, seq)

print(f"Sentence: \"{sentence}\"")
print(f"Tokens:   {tokens_for_display}")
print(f"Attention matrix shape: {avg_attn.shape} (seq_len × seq_len)")

### Visualizing attention

The heatmap below shows which tokens "attend to" which other tokens
(averaged across all attention heads in the last layer). Brighter cells mean
stronger attention. Look for the pronoun *"it"* — it should attend strongly
to *"cat"*, demonstrating coreference resolution.

In [ ]:
# Attention heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(avg_attn, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(tokens_for_display)))
ax.set_yticks(range(len(tokens_for_display)))
ax.set_xticklabels(tokens_for_display, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(tokens_for_display, fontsize=9)
ax.set_xlabel("Attended to (keys)")
ax.set_ylabel("Attending from (queries)")
ax.set_title("Attention patterns — last layer, averaged across heads")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---

## 5 — Generation parameters: your control panel

When you generate text, you have three main knobs that shape the output.
Understanding them mechanically (not just as vibes) is a core AI engineering
skill.

### Temperature

Temperature rescales the logits *before* softmax:

$$P(\text{token}_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **T < 1.0**: sharpens the distribution → model becomes more confident
  and repetitive
- **T = 1.0**: uses the model's raw probabilities
- **T > 1.0**: flattens the distribution → model becomes more random
  and creative

Let's see this visually:

In [ ]:
# Temperature visualization
demo_prompt = "Once upon a time, there was a"
ids = tokenizer(demo_prompt, return_tensors="pt").input_ids.to(model.device)
with torch.no_grad():
    raw_logits = model(ids).logits[0, -1, :].float()

temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 4), sharey=True)

for ax, temp in zip(axes, temperatures):
    scaled_probs = F.softmax(raw_logits / temp, dim=-1)
    top_p, top_i = torch.topk(scaled_probs, 10)
    top_tokens = [tokenizer.decode(i.item()) for i in top_i]
    top_p = top_p.cpu().numpy()

    ax.barh(range(9, -1, -1), top_p[::-1], color="#4C72B0")
    ax.set_yticks(range(9, -1, -1))
    ax.set_yticklabels([repr(t) for t in top_tokens], fontsize=8)
    ax.set_title(f"T = {temp}", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("Token")
fig.suptitle(f'Top-10 token probabilities at different temperatures — "{demo_prompt}..."',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Top-k sampling

Top-k restricts the sampling pool to the **k most likely tokens**, then
renormalizes the probabilities among those k. This prevents the model from
ever selecting wildly improbable tokens.

- **k = 1**: equivalent to greedy decoding
- **k = 50**: a common default, allows moderate diversity
- **k = vocab_size**: no filtering at all

In [ ]:
# Top-k demonstration
probs_raw = F.softmax(raw_logits, dim=-1)

def apply_top_k(probs: torch.Tensor, k: int) -> torch.Tensor:
    """Zero out all but the top-k tokens, then renormalize."""
    top_vals, top_idx = torch.topk(probs, k)
    filtered = torch.zeros_like(probs)
    filtered.scatter_(0, top_idx, top_vals)
    return filtered / filtered.sum()

for k in [1, 5, 50, 1000]:
    filtered = apply_top_k(probs_raw, k)
    nonzero = (filtered > 0).sum().item()
    top_token = tokenizer.decode(torch.argmax(filtered).item())
    entropy = -(filtered[filtered > 0] * filtered[filtered > 0].log()).sum().item()
    print(f"  top-k={k:<5}  tokens in pool: {nonzero:<6}  "
          f"entropy: {entropy:.3f} nats  top: {repr(top_token)}")

### Top-p (nucleus) sampling

Top-p sorts tokens by probability and includes tokens until the cumulative
probability reaches **p**. Unlike top-k, the number of tokens varies
dynamically — confident predictions use fewer tokens, uncertain predictions
use more.

- **p = 0.1**: very focused (only the most probable tokens)
- **p = 0.9**: a common default, good balance
- **p = 1.0**: no filtering

In [ ]:
# Top-p demonstration
def apply_top_p(probs: torch.Tensor, p: float) -> torch.Tensor:
    """Keep tokens until cumulative probability reaches p, then renormalize."""
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    # Create mask: keep tokens where cumulative prob hasn't exceeded p yet
    mask = cumulative - sorted_probs < p
    filtered = torch.zeros_like(probs)
    filtered.scatter_(0, sorted_idx[mask], sorted_probs[mask])
    return filtered / filtered.sum()

print(f"Top-p filtering for: \"{demo_prompt}...\"\n")
for p_val in [0.1, 0.5, 0.9, 0.95, 1.0]:
    filtered = apply_top_p(probs_raw, p_val)
    nonzero = (filtered > 0).sum().item()
    entropy = -(filtered[filtered > 0] * filtered[filtered > 0].log()).sum().item()
    print(f"  top-p={p_val:<5}  tokens in pool: {nonzero:<6}  entropy: {entropy:.3f} nats")

In [ ]:
# Generate from the same prompt with different settings
def generate_text(prompt: str, max_new: int = 40, **kwargs) -> str:
    """Generate text with specified decoding parameters."""
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

story_prompt = "The scientist opened the mysterious box and found"

settings = [
    ("Greedy (T=0)",       dict(do_sample=False)),
    ("Low temp (T=0.3)",   dict(do_sample=True, temperature=0.3, top_p=0.95)),
    ("Medium (T=0.7)",     dict(do_sample=True, temperature=0.7, top_p=0.9)),
    ("High temp (T=1.5)",  dict(do_sample=True, temperature=1.5, top_p=0.95)),
    ("Top-k=5",            dict(do_sample=True, temperature=0.7, top_k=5)),
]

print(f"Prompt: \"{story_prompt}\"\n")
print("=" * 70)
for name, params in settings:
    result = generate_text(story_prompt, **params)
    print(f"\n{name}:")
    print(f"  {result}")
    print("-" * 70)

### Recommended settings for common use cases

| Use case | Temperature | Top-p | Top-k | Notes |
|----------|:-----------:|:-----:|:-----:|-------|
| **Factual Q&A** | 0.0–0.3 | 0.9 | — | Low randomness, stay on topic |
| **Code generation** | 0.0–0.2 | 0.95 | — | Precision matters |
| **Conversational chat** | 0.7 | 0.9 | — | Balance of coherence and variety |
| **Creative writing** | 0.8–1.2 | 0.95 | — | Higher diversity |
| **Brainstorming** | 1.0–1.5 | 0.95 | 50 | Maximum diversity |

> **Practitioner tip**: start with `temperature=0.7, top_p=0.9` and adjust
> from there. Most production systems use these as defaults.

---

## 6 — Mental models for the rest of this course

Everything in AI engineering becomes more intuitive once you see the LLM
as a next-token probability machine. Here's how each major topic connects
back to what we learned today:

### Prompt engineering
You're crafting the **prefix** that steers the probability distribution.
Different prompts produce different attention patterns, which produce
different probability distributions over the next token. Prompt engineering
is the art of finding the prefix that puts the highest probability on the
outputs you want.

### Retrieval-augmented generation (RAG)
You're **injecting context** into the prompt so the model doesn't have to
"remember" facts from its training weights. By placing relevant documents
in the context window, you shift attention toward those facts, making
correct completions more probable.

### Finetuning
You're **adjusting the weights** that determine the probability distribution.
Instead of relying on the right prefix, you modify the model itself so it
naturally assigns higher probability to your desired outputs.

### Agents
You're letting the model's **output become input** in a loop. The model
generates an action (e.g., a function call), you execute it, feed the result
back, and the model generates the next action. Autoregressive generation
meets the real world.

### Evals
You're measuring whether the distribution assigns **high probability to
correct answers**. Whether you're checking exact match, BLEU, or
log-likelihood, you're fundamentally asking: did the model's distribution
peak in the right place?

### Systems
You're making this pipeline — tokenize, embed, transform, predict, sample,
decode — run **fast and cheap at scale**. KV caches, batching, quantization,
speculative decoding — all optimize different steps of the pipeline we traced
today.

In [ ]:
# A visual summary of the pipeline and where each topic fits
pipeline_summary = """
┌─────────────────────────────────────────────────────────────────┐
│                   THE LLM GENERATION PIPELINE                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  "The capital of France is"                                     │
│         │                                                       │
│         ▼                                                       │
│  ┌─────────────┐                                                │
│  │  Tokenize    │ ← Notebook 01                                 │
│  └──────┬──────┘                                                │
│         ▼                                                       │
│  ┌─────────────┐                                                │
│  │  Embed       │ ← Notebook 02                                 │
│  └──────┬──────┘                                                │
│         ▼                                                       │
│  ┌─────────────┐  ← Finetuning adjusts these weights           │
│  │  Transform   │  ← RAG injects context here                   │
│  │  (28 layers) │  ← Prompt engineering steers attention         │
│  └──────┬──────┘  ← Notebook 05                                │
│         ▼                                                       │
│  ┌─────────────┐                                                │
│  │  Predict     │  ← Evals measure this distribution            │
│  └──────┬──────┘                                                │
│         ▼                                                       │
│  ┌─────────────┐                                                │
│  │  Sample      │  ← Notebook 03 (temp, top-k, top-p)           │
│  └──────┬──────┘                                                │
│         ▼                                                       │
│  ┌─────────────┐                                                │
│  │  Decode      │  ← Systems optimize this whole pipeline       │
│  └──────┬──────┘                                                │
│         ▼                                                       │
│  " Paris"                                                       │
│         │                                                       │
│         └──→ Append to input, repeat (autoregressive loop)      │
│              ← Agents loop this with tool use                   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
"""
print(pipeline_summary)

### The one sentence to remember

> **An LLM is a function that takes a sequence of tokens and returns a
> probability distribution over the next token. Everything else in AI
> engineering is about shaping the input, shaping the weights, or
> interpreting the output of that function.**

Keep this mental model as you work through the rest of the course. When
something seems magical, trace it back to the pipeline above.

---

## Exercises

1. **Probe the distribution**: Feed 5 different prompts (a factual question,
   a code snippet, a story opening, a math problem, and a foreign language
   sentence) and examine the top-10 predicted tokens for each. What patterns
   do you notice about the shape of the distribution? When is the model
   confident vs uncertain?

2. **Temperature sweep**: Generate 10 completions each at temperature 0.1,
   0.7, and 1.5 from the same prompt. Count how many *unique* completions
   you get at each setting. Plot the relationship between temperature and
   diversity.

3. **Greedy vs sampling**: Find a prompt where greedy decoding
   (temperature=0) gives a clearly *worse* output than sampling
   (temperature=0.7). Hint: try prompts that benefit from variety, such as
   brainstorming or creative writing. Why does greedy fail here?

In [ ]:
# ── Exercise starter code ──────────────────────────────────────────────

# Exercise 1: Probe the distribution
exercise_prompts = [
    "The largest planet in our solar system is",      # factual
    "def sort_list(arr):\n    return sorted(",        # code
    "In a world where dragons were real,",            # story
    "2 + 2 * 3 =",                                   # math
    "Bonjour, comment",                               # French
]

# TODO: For each prompt, get the top-10 predictions and compare distributions
# Hint: reuse the pattern from Section 1

# Exercise 2: Temperature sweep
# sweep_prompt = "The future of artificial intelligence will"
# TODO: Generate 10 completions at each temperature, count unique ones

# Exercise 3: Greedy vs sampling
# TODO: Find a prompt where greedy decoding produces repetitive/boring output
#       but sampling at T=0.7 produces better text

---

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | An LLM is a function: sequence of tokens → probability distribution over the next token |
| 2 | Generation is a six-step pipeline: tokenize → embed → transform → predict → sample → decode |
| 3 | Next-token prediction forces the model to compress world knowledge into its weights |
| 4 | Attention lets each token incorporate context from every other token in the sequence |
| 5 | Temperature reshapes the probability distribution; top-k and top-p limit the sampling pool |
| 6 | Every AI engineering topic (prompting, RAG, finetuning, agents, evals, systems) maps to a step in this pipeline |

## What's Next

- **Next**: [01_tokenization_deep_dive.ipynb](01_tokenization_deep_dive.ipynb)
  — How text becomes tokens, BPE from scratch, and why tokenization shapes
  everything downstream
- **Related**: [03_sampling_and_decoding.ipynb](03_sampling_and_decoding.ipynb)
  — A deep dive into the sampling strategies we previewed in Section 5

## References & Further Reading

1. [Vaswani et al., "Attention Is All You Need," 2017](https://arxiv.org/abs/1706.03762) — The original transformer paper
2. [Jay Alammar, "The Illustrated Transformer," 2018](https://jalammar.github.io/illustrated-transformer/) — The best visual explanation of transformers
3. [Andrej Karpathy, "Let's build GPT from scratch," 2023](https://www.youtube.com/watch?v=kCc8FmEb1nY) — Hands-on coding walkthrough of a transformer
4. [Holtzman et al., "The Curious Case of Neural Text Degeneration," 2020](https://arxiv.org/abs/1904.09751) — The paper that introduced nucleus (top-p) sampling